In [27]:
"""Install Ultralytics — Kaggle's base image ships PyTorch but not ultralytics."""
!pip install -q ultralytics

# NazarBaan — YOLOv8n Baseline Training

**Project:** NazarBaan ANPR (Pakistani gated-community license plate detection)  
**This notebook:** Trains the first baseline model — YOLOv8n fine-tuned on the Burhan Khan Pk Number Plates dataset.

## Goal

Establish a **reproducible baseline mAP@0.5** for Pakistani license plate detection. This number becomes the benchmark against which all later improvements (dataset merging, hyperparameter tuning, model size upgrades) are measured.

## Training configuration — locked from Phase 2 EDA

| Setting | Value | Reasoning |
|---------|-------|-----------|
| Model | YOLOv8n (nano) | Single class, small dataset — start small, upgrade if needed |
| Image size | **960** | EDA showed 84.8% of plates are <2% of image area; default 640 would lose them |
| Epochs | 100 | With early stopping; we'd rather over-budget than under |
| Patience | 20 | Stop if val mAP plateaus for 20 epochs |
| Batch | 16 | Safe default for T4 16GB |
| Augmentations | YOLOv8 defaults (mosaic + HSV + flips) | EDA suggested mild aug; defaults match |

## Expected outcome

For a clean single-class detection task with ~1.2k images, a healthy baseline lands at **mAP@0.5 ≥ 0.90**. Anything substantially below that signals a data or config problem worth investigating.

In [28]:
"""Verify GPU is live and Ultralytics is the version we expect.
Kaggle preinstalls ultralytics, so we just import and check."""

import torch
import ultralytics
from ultralytics import YOLO

print(f"PyTorch:      {torch.__version__}")
print(f"Ultralytics:  {ultralytics.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM (GB):    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}" if torch.cuda.is_available() else "")

PyTorch:      2.10.0+cu128
Ultralytics:  8.4.52
CUDA:         True
Device:       Tesla T4
VRAM (GB):    15.6


## Step 1 — Patch the dataset config

The dataset was authored on Roboflow's filesystem, so its `data.yaml` has paths like `train: ../train/images` — relative to wherever the YAML lives. On Kaggle our data is mounted **read-only** at `/kaggle/input/nazarbaan-pk-plates-v1/`, so I can't edit that file in place.

The clean fix: **write a new `data.yaml`** under `/kaggle/working/` (writable) that points at the mounted data with absolute paths. YOLO will use this one and ignore the original.

In [29]:
"""Write a Kaggle-aware data.yaml that points at the read-only mount."""

from pathlib import Path
import yaml

DATA_ROOT = Path("/kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-v1")
WORK_DIR = Path("/kaggle/working")

# Read class names from the original yaml — never duplicate them
with open(DATA_ROOT / "data.yaml") as f:
    src = yaml.safe_load(f)

patched = {
    "path": str(DATA_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": src["nc"],
    "names": src["names"],
}

patched_path = WORK_DIR / "data.yaml"
with open(patched_path, "w") as f:
    yaml.safe_dump(patched, f, sort_keys=False)

print(f"Patched data.yaml written to: {patched_path}")
print("---")
print(patched_path.read_text())

Patched data.yaml written to: /kaggle/working/data.yaml
---
path: /kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-v1
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- Number-Plate



## Step 2 — Fine-tune YOLOv8n

I loaded the COCO-pretrained `yolov8n.pt` (Ultralytics auto-downloads ~6 MB on first call) and fine-tune it on our patched dataset. The pretrained weights give me a head start — the backbone already knows what shapes and edges look like; I only need to teach it the concept of "plate."

**`project` and `name` parameters:** all training artifacts go to `runs/detect/baseline_yolov8n/` — weights, plots, confusion matrix, validation predictions. This folder is what I'll download at the end.

Training takes roughly **20–40 minutes** on a T4. The cell streams per-epoch metrics live (precision, recall, mAP50, mAP50-95). Watch the **val mAP50** trend — that's the headline number.

In [30]:
"""Fine-tune YOLOv8n on Pakistani plates."""

model = YOLO("yolov8n.pt")  # COCO-pretrained; will autodownload

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=100,
    patience=20,           # early stop if no mAP improvement for 20 epochs
    imgsz=960,             # higher than default 640 — handles tiny plates
    batch=16,
    project="/kaggle/working/runs/detect",
    name="baseline_yolov8n",
    pretrained=True,
    optimizer="auto",
    seed=42,               # reproducibility
    deterministic=True,
    verbose=True,
    plots=True,            # auto-saves PR curves, confusion matrix, etc.
)

Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

## Step 3 — Evaluate on the test set

The training loop only ever saw the *validation* split for model selection. The **test split** (69 images) is the genuinely held-out set — I touch it exactly once, here, to report the model's real-world generalization. Doing this gives us a number we can quote in the report with a straight face.

In [31]:
"""Run validation against the test split using the best checkpoint."""

best_weights = Path("/kaggle/working/runs/detect/baseline_yolov8n/weights/best.pt")
print(f"Loading best weights: {best_weights}")
print(f"  File size: {best_weights.stat().st_size / 1e6:.1f} MB")

best_model = YOLO(str(best_weights))

test_metrics = best_model.val(
    data="/kaggle/working/data.yaml",
    split="test",
    imgsz=960,
    plots=True,
    project="/kaggle/working/runs/detect",
    name="baseline_yolov8n_test",
)

print("\n=== Test set metrics ===")
print(f"mAP@0.5:      {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {test_metrics.box.map:.4f}")
print(f"Precision:    {test_metrics.box.mp:.4f}")
print(f"Recall:       {test_metrics.box.mr:.4f}")

Loading best weights: /kaggle/working/runs/detect/baseline_yolov8n/weights/best.pt
  File size: 6.3 MB
Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 1.2±0.4 ms, read: 6.0±0.9 MB/s, size: 25.6 KB)
val: Scanning /kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-v1/test/labels... 69 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 69/69 316.8it/s 0.2s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-v1/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.4s0.7s
                   all         69         74      0.963      0.946      0.992      0.782
Speed: 10.0ms preprocess, 8.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/baseline_yolo

## Step 4 — Package training artifacts

I bundle the trained weights, training curves, confusion matrix, and metrics into a single archive so I can download one file and commit it locally with a clean record of what this run produced.

In [32]:
"""Zip up everything we'll want offline: weights, plots, config, metrics."""

import shutil
import json

OUT_DIR = Path("/kaggle/working/baseline_yolov8n_artifacts")
OUT_DIR.mkdir(exist_ok=True)

# Copy the training run folder (weights, plots, args.yaml, results.csv)
shutil.copytree(
    "/kaggle/working/runs/detect/baseline_yolov8n",
    OUT_DIR / "train",
    dirs_exist_ok=True,
)
# Copy the test eval folder (test confusion matrix, PR curves on test)
shutil.copytree(
    "/kaggle/working/runs/detect/baseline_yolov8n_test",
    OUT_DIR / "test",
    dirs_exist_ok=True,
)
# Copy the patched yaml
shutil.copy("/kaggle/working/data.yaml", OUT_DIR / "data.yaml")

# Drop a machine-readable summary
summary = {
    "model": "YOLOv8n",
    "dataset": "burhan-khan/pk-number-plates v1 (1208 images)",
    "imgsz": 960,
    "epochs_configured": 100,
    "patience": 20,
    "batch": 16,
    "test_metrics": {
        "mAP50":      float(test_metrics.box.map50),
        "mAP50_95":   float(test_metrics.box.map),
        "precision":  float(test_metrics.box.mp),
        "recall":     float(test_metrics.box.mr),
    },
}
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Zip the whole bundle for one-click download
archive_path = shutil.make_archive(
    "/kaggle/working/baseline_yolov8n_artifacts", "zip", OUT_DIR
)
print(f"\nArtifact bundle ready: {archive_path}")
print(f"Size: {Path(archive_path).stat().st_size / 1e6:.1f} MB")
print("\nDownload from Kaggle's right sidebar → Output → baseline_yolov8n_artifacts.zip")


Artifact bundle ready: /kaggle/working/baseline_yolov8n_artifacts.zip
Size: 21.4 MB

Download from Kaggle's right sidebar → Output → baseline_yolov8n_artifacts.zip
